# Lab 1: Build Your First GAN

**Goal:** Build a GAN from scratch that learns to generate the number 7 — without ever seeing it.

**What you'll learn:**
- How a Generator and Discriminator work together
- The GAN training loop (the most important thing in this course)
- How to read loss curves and know if training is working
- What happens when you change hyperparameters

**Time:** ~45 minutes

---

## Part 1: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# For reproducibility
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## Part 2: Build the Generator

The Generator's job: take **random noise** and transform it into data that looks real.

```
  Random noise  -->  [Generator]  -->  Fake number
  (1 number)         (neural net)      (trying to be 7)
```

Right now it knows nothing. It will output garbage.

In [ ]:
generator = nn.Sequential(
    nn.Linear(1, 16),     # noise (1 number) -> 16 hidden neurons
    nn.ReLU(),
    nn.Linear(16, 1),     # 16 hidden -> 1 output number
)

print("Generator architecture:")
print(generator)
print(f"\nTotal parameters: {sum(p.numel() for p in generator.parameters())}")

In [ ]:
# Test the untrained Generator
noise = torch.randn(1, 1)       # 1 random number from normal distribution
fake = generator(noise)

print(f"Input noise:        {noise.item():.4f}")
print(f"Generator output:   {fake.item():.4f}")
print(f"\nTarget is 7.0 -- this is far off. Generator hasn't learned yet.")

### Quick check: What does the Generator output for different noise values?

In [ ]:
# Feed 10 different noise values
print("Before training - Generator outputs for different noise:")
print("-" * 40)
with torch.no_grad():
    for i in range(10):
        noise = torch.randn(1, 1)
        output = generator(noise).item()
        print(f"  Noise: {noise.item():+.4f}  -->  Output: {output:+.4f}")

print(f"\nAll random garbage. None close to 7.")

---
## Part 3: Build the Discriminator

The Discriminator's job: look at a number and say **"real" (1.0) or "fake" (0.0)**.

It's a binary classifier — exactly like Lesson 13.

```
  Number  -->  [Discriminator]  -->  Probability (0 to 1)
  (any)        (neural net)          0 = fake, 1 = real
```

In [ ]:
discriminator = nn.Sequential(
    nn.Linear(1, 16),     # input number -> 16 hidden neurons
    nn.ReLU(),
    nn.Linear(16, 1),     # 16 hidden -> 1 output
    nn.Sigmoid(),          # squash to 0-1 probability
)

print("Discriminator architecture:")
print(discriminator)
print(f"\nTotal parameters: {sum(p.numel() for p in discriminator.parameters())}")

In [ ]:
# Test the untrained Discriminator
real_data = torch.tensor([[7.0]])
fake_data = torch.tensor([[2.3]])

print(f"Real data (7.0):  D says {discriminator(real_data).item():.4f}")
print(f"Fake data (2.3):  D says {discriminator(fake_data).item():.4f}")
print(f"\nBoth around 0.5 -- D can't tell the difference yet.")

---
## Part 4: Loss Function and Optimizers

**BCE Loss** (Binary Cross-Entropy) — the same loss we used in Lesson 12 for classification.

- If D should say 1 (real) but says 0.2 → **high loss**
- If D should say 1 (real) and says 0.9 → **low loss**

Each network gets its **own optimizer** because they learn independently.

In [ ]:
loss_fn = nn.BCELoss()

gen_optimizer = torch.optim.Adam(generator.parameters(), lr=0.001)
dis_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.001)

# Labels
real_label = torch.tensor([[1.0]])   # "this is real"
fake_label = torch.tensor([[0.0]])   # "this is fake"

print("Loss function: BCELoss (Binary Cross-Entropy)")
print("Optimizers:    Adam (lr=0.001) for both G and D")

---
## Part 5: The GAN Training Loop

This is the **heart of every GAN**. Each epoch has two steps:

```
Step A: Train the Discriminator
  - Show it REAL data (7.0) --> it should say 1
  - Show it FAKE data (from Generator) --> it should say 0
  - Update D's weights

Step B: Train the Generator
  - Generate fake data
  - Feed it to D
  - G wants D to say 1 (fooled!)
  - Update G's weights
```

**Key insight:** The Generator never sees the real data. It only learns through the Discriminator's feedback.

In [ ]:
# Storage for tracking training progress
gen_losses = []
dis_losses = []
generated_values = []

num_epochs = 2000

for epoch in range(1, num_epochs + 1):

    # =========================================
    # Step A: Train the Discriminator
    # =========================================

    # A1: Real data -- D should say 1
    real_data = torch.tensor([[7.0]])
    real_pred = discriminator(real_data)
    loss_real = loss_fn(real_pred, real_label)

    # A2: Fake data -- D should say 0
    noise = torch.randn(1, 1)
    fake_data = generator(noise).detach()     # .detach() = don't update G here
    fake_pred = discriminator(fake_data)
    loss_fake = loss_fn(fake_pred, fake_label)

    # A3: Update D
    dis_loss = loss_real + loss_fake
    dis_optimizer.zero_grad()
    dis_loss.backward()
    dis_optimizer.step()

    # =========================================
    # Step B: Train the Generator
    # =========================================

    # B1: Generate fake data and get D's opinion
    noise = torch.randn(1, 1)
    fake_data = generator(noise)
    fake_pred = discriminator(fake_data)

    # B2: G wants D to say 1 (real) for the fake data
    gen_loss = loss_fn(fake_pred, real_label)

    # B3: Update G
    gen_optimizer.zero_grad()
    gen_loss.backward()
    gen_optimizer.step()

    # =========================================
    # Track progress
    # =========================================
    gen_losses.append(gen_loss.item())
    dis_losses.append(dis_loss.item())

    # Record what the Generator produces
    with torch.no_grad():
        test_noise = torch.randn(1, 1)
        generated_values.append(generator(test_noise).item())

    # Print milestones
    if epoch in [1, 50, 100, 250, 500, 1000, 1500, 2000]:
        print(f"Epoch {epoch:4d}:  D_loss={dis_loss.item():.4f}  "
              f"G_loss={gen_loss.item():.4f}  "
              f"Generated={generated_values[-1]:.4f}")

print("\nTraining complete!")

---
## Part 6: Visualize the Training

### 6a. Loss curves — the pulse of a GAN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Both losses
axes[0].plot(gen_losses, label='Generator Loss', alpha=0.7, color='blue')
axes[0].plot(dis_losses, label='Discriminator Loss', alpha=0.7, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('G vs D Loss Over Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: What the Generator produces over time
axes[1].plot(generated_values, color='green', alpha=0.5)
axes[1].axhline(y=7.0, color='red', linestyle='--', linewidth=2, label='Target (7.0)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Generated Value')
axes[1].set_title('Generator Output Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6b. How to read these curves

**Healthy GAN training:**
- Both losses fluctuate around similar values (neither dominates)
- Generator output gradually approaches 7.0

**Unhealthy signs:**
- D_loss drops to 0 → Discriminator too strong, Generator can't learn
- G_loss stays flat → Generator is stuck
- Wildly oscillating → learning rate too high

### 6c. Smoothed loss curves (easier to read trends)

In [ ]:
def smooth(values, window=50):
    """Simple moving average."""
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window)
        smoothed.append(sum(values[start:i+1]) / (i - start + 1))
    return smoothed

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(smooth(gen_losses), label='Generator (smoothed)', color='blue', linewidth=2)
ax.plot(smooth(dis_losses), label='Discriminator (smoothed)', color='red', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Smoothed Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 7: Test the Trained Generator

Now let's generate multiple samples and see how close they are to 7.

In [ ]:
print("Generated samples from the trained Generator:")
print("=" * 45)

samples = []
with torch.no_grad():
    for i in range(20):
        noise = torch.randn(1, 1)
        value = generator(noise).item()
        samples.append(value)
        print(f"  Sample {i+1:2d}:  {value:.4f}  "
              f"{'*' * int(max(0, min(50, value * 5)))}")

print(f"\n  Mean:   {sum(samples)/len(samples):.4f}")
print(f"  Target: 7.0000")
print(f"  Error:  {abs(7.0 - sum(samples)/len(samples)):.4f}")

In [ ]:
# Histogram of 1000 generated samples
with torch.no_grad():
    noise = torch.randn(1000, 1)
    all_samples = generator(noise).squeeze().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_samples, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=7.0, color='red', linestyle='--', linewidth=2, label='Target (7.0)')
ax.set_xlabel('Generated Value')
ax.set_ylabel('Count')
ax.set_title('Distribution of 1000 Generated Samples')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean: {all_samples.mean():.4f}")
print(f"Std:  {all_samples.std():.4f}")

---
## Part 8: Understand the Key Concepts

### 8a. Why `.detach()`?

When training D, we use the Generator to create fake data but we **don't want to update the Generator**.

`.detach()` breaks the computation graph — gradients stop flowing back to G.

In [ ]:
# Demonstration: detach breaks the gradient chain
noise = torch.randn(1, 1)

# WITH detach: G's parameters won't get gradients
fake_detached = generator(noise).detach()
print(f"With .detach():    requires_grad = {fake_detached.requires_grad}")

# WITHOUT detach: G's parameters WILL get gradients
fake_connected = generator(noise)
print(f"Without .detach(): requires_grad = {fake_connected.requires_grad}")

print("\nRule: Use .detach() when training D. Don't use it when training G.")

### 8b. Why does the Generator use `real_label`?

```python
gen_loss = loss_fn(fake_pred, real_label)   # G wants D to say 1!
```

The Generator's goal is to **fool** D. Success = D says "real" (1.0) for fake data.

So G's loss is: "how far is D's prediction from 1.0 for my fake data?"

In [ ]:
# The adversarial game in numbers
with torch.no_grad():
    noise = torch.randn(1, 1)
    fake = generator(noise)
    d_opinion = discriminator(fake).item()
    d_on_real = discriminator(torch.tensor([[7.0]])).item()

print("The adversarial game right now:")
print(f"  D on real data (7.0):    {d_on_real:.4f}  (should be ~1.0)")
print(f"  D on fake data ({fake.item():.2f}):  {d_opinion:.4f}  (D says fake=0, G wants 1)")
print(f"\n  Generator is {'winning' if d_opinion > 0.5 else 'losing'} "
      f"(D {'fooled' if d_opinion > 0.5 else 'not fooled'})")

---
## Part 9: Experiments

Now it's your turn. Run each experiment, observe what changes, and answer the questions.

### Experiment 1: Change the target number

In [ ]:
def train_gan_for_target(target, epochs=2000):
    """Train a GAN to generate a specific target number."""
    G = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
    D = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())

    loss_fn = nn.BCELoss()
    g_opt = torch.optim.Adam(G.parameters(), lr=0.001)
    d_opt = torch.optim.Adam(D.parameters(), lr=0.001)

    real_label = torch.tensor([[1.0]])
    fake_label = torch.tensor([[0.0]])
    history = []

    for epoch in range(1, epochs + 1):
        # Train D
        real_data = torch.tensor([[target]])
        d_loss = loss_fn(D(real_data), real_label) + \
                 loss_fn(D(G(torch.randn(1, 1)).detach()), fake_label)
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        # Train G
        g_loss = loss_fn(D(G(torch.randn(1, 1))), real_label)
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

        with torch.no_grad():
            history.append(G(torch.randn(1, 1)).item())

    return history

# Train for different targets
targets = [3.0, 7.0, 15.0, -5.0]
results = {}

for t in targets:
    print(f"Training for target = {t}...")
    results[t] = train_gan_for_target(t)

print("Done!")

In [ ]:
# Visualize: how does each GAN learn its target?
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['blue', 'green', 'orange', 'purple']
for (target, history), color in zip(results.items(), colors):
    ax.plot(history, alpha=0.4, color=color)
    ax.axhline(y=target, linestyle='--', color=color, linewidth=2,
               label=f'Target = {target}')

ax.set_xlabel('Epoch')
ax.set_ylabel('Generated Value')
ax.set_title('Experiment 1: GAN Learning Different Targets')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final values:")
for target, history in results.items():
    print(f"  Target {target:+6.1f}  -->  Generated {history[-1]:+.4f}  "
          f"(error: {abs(target - history[-1]):.4f})")

**Question:** Which target was hardest to learn? Why do you think that is?

### Experiment 2: What happens with different learning rates?

In [ ]:
def train_gan_with_lr(lr_g, lr_d, epochs=2000):
    """Train a GAN with specified learning rates."""
    G = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
    D = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())

    loss_fn = nn.BCELoss()
    g_opt = torch.optim.Adam(G.parameters(), lr=lr_g)
    d_opt = torch.optim.Adam(D.parameters(), lr=lr_d)

    real_label = torch.tensor([[1.0]])
    fake_label = torch.tensor([[0.0]])
    g_losses, d_losses, values = [], [], []

    for epoch in range(1, epochs + 1):
        real_data = torch.tensor([[7.0]])
        d_loss = loss_fn(D(real_data), real_label) + \
                 loss_fn(D(G(torch.randn(1, 1)).detach()), fake_label)
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        g_loss = loss_fn(D(G(torch.randn(1, 1))), real_label)
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())
        with torch.no_grad():
            values.append(G(torch.randn(1, 1)).item())

    return g_losses, d_losses, values

# Different LR combinations
experiments = {
    'Balanced (G=0.001, D=0.001)':    (0.001, 0.001),
    'Fast D (G=0.001, D=0.01)':       (0.001, 0.01),
    'Fast G (G=0.01, D=0.001)':       (0.01, 0.001),
    'Both slow (G=0.0001, D=0.0001)': (0.0001, 0.0001),
}

lr_results = {}
for name, (lr_g, lr_d) in experiments.items():
    print(f"Training: {name}...")
    lr_results[name] = train_gan_with_lr(lr_g, lr_d)

print("Done!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, (g_losses, d_losses, values)) in zip(axes.flat, lr_results.items()):
    ax.plot(values, alpha=0.5, color='green')
    ax.axhline(y=7.0, color='red', linestyle='--', linewidth=2)
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Generated Value')
    ax.set_ylim(-5, 15)
    ax.grid(True, alpha=0.3)
    ax.text(0.95, 0.05, f'Final: {values[-1]:.2f}',
            transform=ax.transAxes, ha='right', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Experiment 2: Effect of Learning Rates', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Questions:**
1. What happens when D's learning rate is much higher than G's?
2. What happens when both are very slow?
3. Which combination worked best? Why?

### Experiment 3: What happens with different network sizes?

In [ ]:
def train_gan_with_size(hidden_size, epochs=2000):
    """Train a GAN with a given hidden layer size."""
    G = nn.Sequential(nn.Linear(1, hidden_size), nn.ReLU(), nn.Linear(hidden_size, 1))
    D = nn.Sequential(nn.Linear(1, hidden_size), nn.ReLU(),
                      nn.Linear(hidden_size, 1), nn.Sigmoid())

    loss_fn = nn.BCELoss()
    g_opt = torch.optim.Adam(G.parameters(), lr=0.001)
    d_opt = torch.optim.Adam(D.parameters(), lr=0.001)

    real_label = torch.tensor([[1.0]])
    fake_label = torch.tensor([[0.0]])
    values = []

    for epoch in range(1, epochs + 1):
        real_data = torch.tensor([[7.0]])
        d_loss = loss_fn(D(real_data), real_label) + \
                 loss_fn(D(G(torch.randn(1, 1)).detach()), fake_label)
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        g_loss = loss_fn(D(G(torch.randn(1, 1))), real_label)
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

        with torch.no_grad():
            values.append(G(torch.randn(1, 1)).item())

    return values

sizes = [4, 16, 64, 256]
size_results = {}

for size in sizes:
    print(f"Training with hidden_size = {size}...")
    size_results[size] = train_gan_with_size(size)

print("Done!")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['red', 'blue', 'green', 'purple']
for (size, values), color in zip(size_results.items(), colors):
    ax.plot(values, alpha=0.5, color=color, label=f'hidden={size}')

ax.axhline(y=7.0, color='black', linestyle='--', linewidth=2, label='Target (7.0)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Generated Value')
ax.set_title('Experiment 3: Effect of Network Size')
ax.legend()
ax.set_ylim(-5, 15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final values:")
for size, values in size_results.items():
    params = 2 * (1 * size + size + size * 1 + 1)  # rough param count
    print(f"  hidden={size:3d}: Generated {values[-1]:+.4f}  (~{params} params)")

**Question:** Does a bigger network always converge faster? Is there a sweet spot?

### Experiment 4: Train on a distribution, not a single number

Real GANs learn **distributions**, not single values. Let's make our GAN learn a Gaussian distribution centered at 7.

In [ ]:
# Train GAN to learn a Gaussian distribution: mean=7, std=1
G = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))
D = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

loss_fn = nn.BCELoss()
g_opt = torch.optim.Adam(G.parameters(), lr=0.001)
d_opt = torch.optim.Adam(D.parameters(), lr=0.001)

snapshots = {}  # save generated distributions at different epochs

for epoch in range(1, 5001):
    batch_size = 64

    # Real data: samples from N(7, 1)
    real_data = torch.randn(batch_size, 1) + 7.0  # Gaussian centered at 7
    real_labels = torch.ones(batch_size, 1)
    fake_labels = torch.zeros(batch_size, 1)

    # Train D
    noise = torch.randn(batch_size, 1)
    fake_data = G(noise).detach()
    d_loss = loss_fn(D(real_data), real_labels) + loss_fn(D(fake_data), fake_labels)
    d_opt.zero_grad(); d_loss.backward(); d_opt.step()

    # Train G
    noise = torch.randn(batch_size, 1)
    fake_data = G(noise)
    g_loss = loss_fn(D(fake_data), real_labels)
    g_opt.zero_grad(); g_loss.backward(); g_opt.step()

    # Save snapshots
    if epoch in [1, 100, 500, 1000, 3000, 5000]:
        with torch.no_grad():
            test_noise = torch.randn(1000, 1)
            snapshots[epoch] = G(test_noise).squeeze().numpy()
        print(f"Epoch {epoch:4d}: D_loss={d_loss.item():.4f}  G_loss={g_loss.item():.4f}")

print("Done!")

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# True distribution for comparison
x_range = np.linspace(2, 12, 200)
true_pdf = np.exp(-0.5 * (x_range - 7)**2) / np.sqrt(2 * np.pi)

for ax, (epoch, samples) in zip(axes.flat, snapshots.items()):
    ax.hist(samples, bins=50, density=True, alpha=0.6, color='steelblue',
            edgecolor='white', label='Generated')
    ax.plot(x_range, true_pdf, 'r-', linewidth=2, label='True N(7,1)')
    ax.set_title(f'Epoch {epoch}', fontsize=12)
    ax.set_xlim(2, 12)
    ax.set_ylim(0, 0.8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('GAN Learning a Gaussian Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**This is the real insight:** GANs don't just memorize one number — they learn the **shape of data distributions**. This is what makes them powerful for generating images.

---

## Part 10: Summary

| Concept | What you learned |
|---|---|
| Generator | Takes noise, produces fake data |
| Discriminator | Binary classifier: real (1) or fake (0) |
| `.detach()` | Blocks gradients when training D |
| Adversarial | G and D have opposite goals |
| Loss curves | Both losses should stay similar — neither should dominate |
| Learning rate | Balance between G and D is critical |
| Distributions | GANs learn data distributions, not single values |

**Next lab:** We scale this up from numbers to **images** — generating handwritten digits with MNIST.